In [1]:
import os
import sys
sys.path.append(os.path.abspath(".."))
import pandas as pd
import torch
from preprocessing import *
from RNN import RNN
import time
set_seed(42)


In [2]:
df = pd.read_csv(r'../output_csv/ANL-Intrepid-2009-1.swf.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12431 entries, 0 to 12430
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   job_id                    12431 non-null  float64
 1   submit_time               12431 non-null  float64
 2   wait_time                 12430 non-null  float64
 3   run_time                  12430 non-null  float64
 4   num_allocated_processors  12430 non-null  float64
 5   avg_cpu_time_used         12430 non-null  float64
 6   used_memory               12430 non-null  float64
 7   requested_processors      12430 non-null  float64
 8   requested_time            12430 non-null  float64
 9   requested_memory          12430 non-null  float64
 10  status                    12430 non-null  float64
 11  user_id                   12430 non-null  float64
 12  group_id                  12430 non-null  float64
 13  executable_id             12430 non-null  float64
 14  queue_

In [3]:
feature_columns = ['requested_processors', 'requested_time', 'submit_time', 'wait_time', 'user_id', 'queue_id']
target_column = 'run_time'

# Hyperparameters
input_dim = len(feature_columns)
num_hidden = 64
num_layers = 3
dropout = 0.2
num_epochs = 10
lr = 0.001
batch_size = 128  
seq_len = 20
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dataloaders, scaler = prepare_data_seq(df, feature_columns, target_column, statuss=-1, seq_len=seq_len, batch_size=batch_size)

In [4]:
import os, sys
sys.path.append(os.path.abspath(".."))

rmse_lst, mae_lst, mse_lst, r2_lst, infer_time_lst = [], [], [], [], []

for iter in range(2):
    print(f'\nIteration {iter + 1} / 2')
    model = RNN(input_dim, num_hidden, num_layers, dropout).to(device)
    
    start = time.time()
    model.train_model(model, dataloaders['train'], dataloaders['val'], epochs=num_epochs, lr=lr, scaler=scaler, save_path='../models_ANL/best_rnn_model.pth')
    train_time = time.time() - start
    
    model.load_state_dict(torch.load('../models_ANL/best_rnn_model.pth', map_location=device))
    
    start = time.time()
    rmse, mae, mse, r2 = model.evaluate_model(model, dataloaders['test'], scaler, num_features=input_dim)
    infer_time = (time.time() - start) / len(dataloaders['test'].dataset)
    
    rmse_lst.append(rmse); mae_lst.append(mae); mse_lst.append(mse); r2_lst.append(r2); infer_time_lst.append(infer_time)
    print(f'Test RMSE: {rmse:.4f}, R2: {r2:.4f}, Training Time: {train_time:.2f}s')

import pandas as pd
results_df = pd.DataFrame({'RMSE': rmse_lst, 'MAE': mae_lst, 'MSE': mse_lst, 'R2': r2_lst, 'Inference Time': infer_time_lst})
results_df.to_csv('../output_ANL/rnn_results_new.csv', index=False)



Iteration 1 / 2
Epoch [1/10], Train Loss: 0.1761, Val Loss: 0.1404
Val RMSE: 5132.0300, Val MAE: 2383.3146, Val MSE: 26337731.4698, Val R2: 0.6031
Model saved at epoch 1 with validation loss: 0.1404
Epoch [2/10], Train Loss: 0.1450, Val Loss: 0.1332
Val RMSE: 4961.4732, Val MAE: 2177.5589, Val MSE: 24616216.1407, Val R2: 0.6291
Model saved at epoch 2 with validation loss: 0.1332
Epoch [3/10], Train Loss: 0.1419, Val Loss: 0.1334
Val RMSE: 4960.3814, Val MAE: 2299.1110, Val MSE: 24605383.6592, Val R2: 0.6292
Epoch [4/10], Train Loss: 0.1395, Val Loss: 0.1298
Val RMSE: 4880.4556, Val MAE: 2161.3846, Val MSE: 23818846.6766, Val R2: 0.6411
Model saved at epoch 4 with validation loss: 0.1298
Epoch [5/10], Train Loss: 0.1375, Val Loss: 0.1346
Val RMSE: 5034.5767, Val MAE: 2229.3997, Val MSE: 25346962.3643, Val R2: 0.6181
Epoch [6/10], Train Loss: 0.1357, Val Loss: 0.1262
Val RMSE: 4850.5825, Val MAE: 2117.6554, Val MSE: 23528150.8631, Val R2: 0.6455
Model saved at epoch 6 with validation lo

/tmp/ipykernel_1213741/3905623836.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('../models_ANL/best_rnn_model.pth', map_location=devi

Epoch [1/10], Train Loss: 0.1731, Val Loss: 0.1383
Val RMSE: 5021.8649, Val MAE: 2465.4254, Val MSE: 25219127.5289, Val R2: 0.6200
Model saved at epoch 1 with validation loss: 0.1383
Epoch [2/10], Train Loss: 0.1463, Val Loss: 0.1369
Val RMSE: 5046.7964, Val MAE: 2230.8034, Val MSE: 25470154.0681, Val R2: 0.6162
Model saved at epoch 2 with validation loss: 0.1369
Epoch [3/10], Train Loss: 0.1434, Val Loss: 0.1357
Val RMSE: 5052.4344, Val MAE: 2204.1126, Val MSE: 25527093.8070, Val R2: 0.6153
Model saved at epoch 3 with validation loss: 0.1357
Epoch [4/10], Train Loss: 0.1396, Val Loss: 0.1382
Val RMSE: 5121.9505, Val MAE: 2240.5296, Val MSE: 26234377.2548, Val R2: 0.6047
Epoch [5/10], Train Loss: 0.1376, Val Loss: 0.1276
Val RMSE: 4863.9088, Val MAE: 2083.4658, Val MSE: 23657608.6632, Val R2: 0.6435
Model saved at epoch 5 with validation loss: 0.1276
Epoch [6/10], Train Loss: 0.1351, Val Loss: 0.1292
Val RMSE: 4925.6793, Val MAE: 2129.7574, Val MSE: 24262316.6195, Val R2: 0.6344
Epoch 

/tmp/ipykernel_1213741/3905623836.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('../models_ANL/best_rnn_model.pth', map_location=devi

Test RMSE: 4727.1771, R2: 0.6175, Training Time: 18.27s
